# Field Shift · Punto 3: el "por qué" en lenguaje simple

**Qué ve el productor** (pantalla "Próxima campaña" / "Compare rotations"): debajo de cada opción de rotación, dos o tres frases cortas en inglés que dicen qué empujó el rinde de su condado para arriba o para abajo, con su fuente. Por ejemplo:

> "In years with a hot July (corn flowering), corn yields here tend to be about 25 bu/ac lower (NASA POWER)."
> "Corn after corn tends to yield about 17% less than corn after soybean (USDA CDL + NASS, 2009–2023)."

**Qué hace:** toma el modelo del punto 2 (modelo B, escenarios de rotación) y lo explica. Hay **dos fuentes de explicación que no se mezclan** (`PUNTO2.md`, sección "Para el punto 3"):

1. **Clima y suelo:** valores SHAP del LightGBM del punto 2, sumados por grupo de variables y traducidos con plantillas fijas.
2. **Rotación:** el efecto medido con efectos fijos en el punto 2 (`backend/Datos/efecto_rotacion.csv`). La rotación **nunca** se explica con SHAP, porque en el modelo de ML esas columnas miden qué tipo de condado es, no el efecto de rotar.

No hay texto generado por IA: todas las frases salen de plantillas fijas, así no se inventa nada y no hay nada que declarar en Space Apps. El modelo estima asociaciones; la decisión es del productor.

**Datos:** `backend/Datos/tabla_modelado.csv`, `efecto_rotacion.csv` y `escenarios_rotacion_resumen.csv`, que deja el punto 2 (están en Git). Los modelos `backend/ML/modelos/punto2_rotaciones_<cultivo>.joblib` no están en Git: si no están, esta notebook los **regenera con la receta exacta del punto 2** (paso 13) a partir de `tabla_modelado.csv`, y lo avisa.

**Cómo usarla:** correr todo de arriba hacia abajo (Kernel → Restart & Run All). Procesa maíz y soja en la misma corrida.

## 0. Librerías

- **pandas / numpy:** tablas y cuentas.
- **LightGBM:** el modelo de árboles del punto 2. Con `pred_contrib=True` devuelve los valores SHAP exactos, sin instalar `shap`.
- **scikit-learn:** solo para rearmar el modelo si falta el `.joblib`.
- **joblib:** carga el modelo guardado.

**SHAP, en simple:** para una predicción, reparte la diferencia respecto del promedio entre las variables. Cada variable recibe un número en bu/ac: positivo si empujó el rinde para arriba, negativo si para abajo. Sumados, dan exactamente la predicción.

In [1]:
import datetime
import json
import re
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
try:
    import lightgbm as lgb
    USAR_LGBM = True
except ImportError:  # equivalente de scikit-learn, igual que en el punto 2
    USAR_LGBM = False
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

## 1. Parámetros

Todo lo que se puede cambiar está acá:

- cultivos a explicar y rutas;
- **grupos de variables**: al productor no le sirve "`gwetroot_05`", así que el SHAP se suma por grupo (los de `PUNTO2.md`) y hay una frase fija por grupo;
- cuántas razones por opción y desde qué tamaño (en bu/ac) una razón vale la pena mostrarse;
- qué fracción de los años cuenta como "años malos" para cada factor del clima.

El grupo `tipo_de_condado` (las columnas de rotación del modelo de ML) **no se muestra**: mide qué tipo de condado es, no el efecto de rotar.

In [2]:
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
SALIDA = RAIZ / "backend" / "Datos"
FRONT = SALIDA / "front"
CARPETA_MODELOS = RAIZ / "backend" / "ML" / "modelos"

CULTIVOS = ["maiz", "soja"]
TARGET = "rend_bu_ac"
ANIOS_PRUEBA = [2024, 2025]      # la prueba del punto 2 sigue sin tocarse: acá solo se explican 2009–2023
SEMILLA = 42
NOMBRES_ESTADO = {"17": "Illinois", "18": "Indiana", "19": "Iowa", "20": "Kansas", "27": "Minnesota",
                  "29": "Missouri", "31": "Nebraska", "38": "North Dakota", "39": "Ohio", "46": "South Dakota", "55": "Wisconsin"}

N_RAZONES = 3                                  # razones por opción (el front muestra 2 o 3)
UMBRAL_BU_AC = {"maiz": 2.0, "soja": 1.0}      # una razón entra si mueve al menos esto (bu/ac)
MIN_CAMBIO_BU_AC = 0.5                         # si rotar cambia menos que esto, no se escribe "+0 bu/ac"
FRACCION_ANIOS_MALOS = 0.3                     # "años malos" de un factor = el 30 % de años con SHAP más negativo
MIN_ANIOS_MALOS = 2

# Grupos de variables (PUNTO2.md, "Grupos de variables y frases"). <c> se reemplaza por el cultivo.
GRUPOS = {
    "calor_julio": ["tmax_jul_c", "dias_tmax_gt30_jul", "dias_tmax_gt35_07"],
    "agua_julio": ["prec_jul_mm", "deficit_mm_07"],
    "calor_agosto": ["tmax_c_08"],
    "agua_agosto": ["prec_ago_mm", "deficit_mm_08"],
    "temporada": ["prec_temporada_mm", "gdd_temporada", "deficit_temporada_mm"],
    "primavera_humeda": ["prec_power_mm_04", "prec_power_mm_05", "dias_lluvia_gt50_04", "dias_lluvia_gt50_05",
                         "gwettop_04", "gwettop_05", "gwetroot_04", "gwetroot_05"],
    "suelo_productivo": ["nccpi_<c>", "soc0_30_gm2"],
    "agua_del_suelo": ["aws0_100_mm", "clase_drenaje", "grupo_hidrologico"],
    "terreno": ["pendiente_media_pct", "twi_medio"],
    "tipo_de_condado": ["pct_<c>_tras_<c>_media", "pct_<c>_tras_<c>_desvio"],
}
GRUPOS_CLIMA = ["calor_julio", "agua_julio", "calor_agosto", "agua_agosto", "temporada", "primavera_humeda"]
GRUPOS_SUELO = ["suelo_productivo", "agua_del_suelo", "terreno"]
NO_MOSTRAR = ["tipo_de_condado"]

# Variable que representa a cada grupo del clima, para decir si el año malo fue "seco" o "húmedo", "caluroso" o "fresco".
REPRESENTANTE = {
    "calor_julio": ("tmax_jul_c", "calor"), "agua_julio": ("deficit_mm_07", "agua"),
    "calor_agosto": ("tmax_c_08", "calor"), "agua_agosto": ("deficit_mm_08", "agua"),
    "temporada": ("deficit_temporada_mm", "agua"), "primavera_humeda": ("prec_power_mm_05", "agua"),
}

# Fuentes: ids de fuentes.json (INFO.md 4.5) y la etiqueta corta que se muestra.
FUENTES = {"power": "NASA POWER", "ssurgo": "USDA SSURGO", "nasadem": "NASA NASADEM",
           "cdl": "USDA CDL + NASS", "nass": "USDA NASS"}
GRUPO_FUENTE = {**{g: "power" for g in GRUPOS_CLIMA}, "suelo_productivo": "ssurgo", "agua_del_suelo": "ssurgo",
                "terreno": "nasadem", "tipo_de_condado": "cdl"}

CULTIVO_EN = {"maiz": "corn", "soja": "soybean"}
OTRO_CULTIVO = {"maiz": "soja", "soja": "maiz"}   # con qué rota cada uno en estas regiones
PROHIBIDAS = ("because", "recommend", "best option", "you should", "guarantee", "will yield")

HAY_DATOS = all((SALIDA / f).exists() for f in ["tabla_modelado.csv", "efecto_rotacion.csv", "escenarios_rotacion_resumen.csv"])
print("¿Están las salidas del punto 2?", HAY_DATOS)
if not HAY_DATOS:
    print("Falta alguna salida del punto 2 en backend/Datos/. La notebook queda en espera.")

¿Están las salidas del punto 2? True


## 2. Cargar lo que dejó el punto 2

- `tabla_modelado.csv`: la misma tabla que usó el modelo (una fila por condado y año, con el rinde y las variables crudas).
- `efecto_rotacion.csv`: cuánto cambia el rinde por punto de "cultivo sobre sí mismo", medido con efectos fijos.
- `escenarios_rotacion_resumen.csv`: la mediana entre climas de cada opción (`actual`, `rotado`, `continuo`) por condado.
- `experimentos.csv`: de ahí sale la lista exacta de features del modelo elegido.

`fips` se lee como texto para no perder el cero de la izquierda.

In [3]:
if HAY_DATOS:
    tabla_todo = pd.read_csv(SALIDA / "tabla_modelado.csv", dtype={"fips": str, "estado": str})
    efecto = pd.read_csv(SALIDA / "efecto_rotacion.csv").set_index("cultivo")
    resumen_esc = pd.read_csv(SALIDA / "escenarios_rotacion_resumen.csv", dtype={"fips": str})
    experimentos = pd.read_csv(SALIDA / "experimentos.csv")
    print("tabla_modelado:", tabla_todo.shape, "| condados:", tabla_todo["fips"].nunique(),
          "| estados:", sorted(tabla_todo["estado"].map(NOMBRES_ESTADO).unique()))
    display(efecto[["columna", "bu_ac_por_punto", "ic95_bajo", "ic95_alto", "continuo_vs_rotado_pct", "n"]])
else:
    print("En espera de los datos.")

tabla_modelado: (25492, 34) | condados: 948 | estados: ['Illinois', 'Indiana', 'Iowa', 'Kansas', 'Minnesota', 'Missouri', 'Nebraska', 'North Dakota', 'Ohio', 'South Dakota', 'Wisconsin']


,columna,bu_ac_por_punto,ic95_bajo,ic95_alto,continuo_vs_rotado_pct,n
cultivo,,,,,,
maiz,pct_maiz_tras_maiz,-0.3308,-0.4289,-0.2328,-18.9,8449
soja,pct_soja_tras_soja,-0.0617,-0.0921,-0.0314,-12.1,8425


## 3. El modelo del punto 2: cargarlo o regenerarlo

Los `.joblib` no van a Git. Si están en `backend/ML/modelos/`, se cargan. Si no, se **reentrena el modelo final con la receta exacta del paso 13 del punto 2** (mismas features, mismos hiperparámetros, misma semilla, años 2009–2023) usando `tabla_modelado.csv`. Da el mismo modelo que guardó el punto 2 con la misma versión de LightGBM; se avisa cuál de las dos cosas pasó.

Las funciones de abajo son copia de las del punto 2 (`ajustar_tendencia`, `aplicar_tendencia`, `separar_rotacion`, `crear_modelo`). Si cambian allá, hay que cambiarlas acá.

In [4]:
TENDENCIA = "condado"   # igual que el punto 2: la recta del estado + el nivel propio de cada condado
ENCOGER_NIVEL = 2


def ajustar_tendencia(tr, modo=None):
    """Recta rinde ~ año por estado, solo con el train (copia del punto 2). Con modo "condado" suma el nivel propio
    de cada condado (cuánto rinde por encima o por debajo de la recta de su estado), achicado hacia 0 si tiene pocos años."""
    modo = modo or TENDENCIA
    por_estado = {e: np.polyfit(g["anio"], g[TARGET], 1) for e, g in tr.groupby("estado")}
    coefs = {"por_estado": por_estado, "global": np.polyfit(tr["anio"], tr[TARGET], 1)}
    if modo == "condado":
        resto = (tr[TARGET] - aplicar_tendencia(coefs, tr)).groupby(tr["fips"])
        coefs["nivel_condado"] = (resto.sum() / (resto.size() + ENCOGER_NIVEL)).to_dict()
    return coefs


def aplicar_tendencia(coefs, datos, anio=None):
    """Valor de la tendencia para cada fila. Un condado sin nivel propio usa solo la recta de su estado."""
    anios = datos["anio"] if anio is None else pd.Series(anio, index=datos.index)
    valor = np.array([np.polyval(coefs["por_estado"].get(e, coefs["global"]), a)
                      for e, a in zip(datos["estado"], anios)])
    if "nivel_condado" in coefs:
        valor = valor + datos["fips"].map(coefs["nivel_condado"]).fillna(0).to_numpy()
    return valor


def separar_rotacion(datos, medias, rotacion):
    """Agrega <pct>_media y <pct>_desvio (copia del punto 2)."""
    d = datos.copy()
    for c in rotacion:
        media = d["fips"].map(medias[c]).fillna(d.groupby("fips")[c].transform("mean"))
        d[f"{c}_media"] = media
        d[f"{c}_desvio"] = d[c] - media
    return d


def crear_modelo_lgbm(num, cat):
    """El LightGBM poco profundo del punto 2 (paso 7)."""
    prep = ColumnTransformer([
        ("num", "passthrough", num),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1), cat),
    ])
    if USAR_LGBM:
        arboles = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, max_depth=3, num_leaves=7,
                                    min_child_samples=40, subsample=0.8, subsample_freq=1,
                                    colsample_bytree=0.8, random_state=SEMILLA, verbose=-1)
    else:
        arboles = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, max_depth=3,
                                                min_samples_leaf=40, random_state=SEMILLA)
    return Pipeline([("prep", prep), ("modelo", arboles)])


def cargar_o_regenerar(cultivo, tabla):
    ruta = CARPETA_MODELOS / f"punto2_rotaciones_{cultivo}.joblib"
    if ruta.exists():
        paquete = joblib.load(ruta)
        paquete["origen"] = f"cargado de {ruta.relative_to(RAIZ)}"
        return paquete
    # Regenerar con la receta del punto 2
    exp = experimentos[(experimentos["cultivo"] == cultivo) & (experimentos["modelo"] == "lightgbm")]
    if exp.empty:
        raise RuntimeError(f"No hay corrida de lightgbm para {cultivo} en experimentos.csv")
    features = exp.iloc[-1]["features"].split(";")
    rotacion = [f"pct_{cultivo}_tras_{cultivo}"]
    cat = [c for c in ["clase_drenaje", "grupo_hidrologico"] if c in features]
    num = [c for c in features if c not in cat]
    dev = tabla[~tabla["anio"].isin(ANIOS_PRUEBA)]
    paquete = {"tendencia": ajustar_tendencia(dev), "medias": dev.groupby("fips")[rotacion].mean(),
               "features": features, "rotacion": rotacion, "cultivo": cultivo, "target": TARGET}
    x = separar_rotacion(dev, paquete["medias"], rotacion)[features]
    paquete["modelo"] = crear_modelo_lgbm(num, cat).fit(x, dev[TARGET] - aplicar_tendencia(paquete["tendencia"], dev))
    paquete["origen"] = "regenerado desde tabla_modelado.csv con la receta del punto 2 (no estaba el .joblib)"
    return paquete


MODELOS, TABLAS = {}, {}
if HAY_DATOS:
    for cultivo in CULTIVOS:
        TABLAS[cultivo] = tabla_todo[tabla_todo["cultivo"] == cultivo].reset_index(drop=True)
        MODELOS[cultivo] = cargar_o_regenerar(cultivo, TABLAS[cultivo])
        print(f"{cultivo}: {MODELOS[cultivo]['origen']} | {len(MODELOS[cultivo]['features'])} features")
else:
    print("En espera de los datos.")

maiz: cargado de backend/ML/modelos/punto2_rotaciones_maiz.joblib | 25 features
soja: cargado de backend/ML/modelos/punto2_rotaciones_soja.joblib | 23 features


## 4. Valores SHAP por condado y año

Para cada fila de desarrollo (2009–2023) se pide a LightGBM la contribución de cada variable (`pred_contrib=True`). Se verifica que **base + suma de contribuciones = predicción**: si no da, algo está mal y la notebook se frena.

Los valores están en **bu/ac** y explican la **anomalía** (rinde − tendencia del estado), no el rinde total. Positivo = empujó para arriba.

In [5]:
def shap_del_modelo(paquete, datos):
    """DataFrame de contribuciones (una columna por variable, en bu/ac) y la base."""
    x = separar_rotacion(datos, paquete["medias"], paquete["rotacion"])[paquete["features"]]
    prep, arboles = paquete["modelo"].named_steps["prep"], paquete["modelo"].named_steps["modelo"]
    xt = prep.transform(x)
    nombres = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]
    if hasattr(arboles, "booster_"):
        contrib = arboles.booster_.predict(xt, pred_contrib=True)
    elif type(arboles).__name__ == "Ridge":
        raise RuntimeError("El modelo elegido es Ridge: usar coeficiente × valor escalado (ver PUNTO2.md).")
    else:
        import shap  # solo si el modelo no es LightGBM
        ex = shap.TreeExplainer(arboles)
        vals = ex.shap_values(xt)
        contrib = np.column_stack([vals, np.full(len(xt), ex.expected_value)])
    shap_df = pd.DataFrame(contrib[:, :-1], columns=nombres, index=datos.index)
    base = float(contrib[0, -1])
    pred = paquete["modelo"].predict(x)
    assert np.allclose(base + shap_df.sum(axis=1), pred, atol=1e-5), "SHAP no suma a la predicción"
    return shap_df, base


SHAP, BASE, DEV = {}, {}, {}
if HAY_DATOS:
    for cultivo in CULTIVOS:
        DEV[cultivo] = TABLAS[cultivo][~TABLAS[cultivo]["anio"].isin(ANIOS_PRUEBA)].reset_index(drop=True)
        SHAP[cultivo], BASE[cultivo] = shap_del_modelo(MODELOS[cultivo], DEV[cultivo])
        print(f"{cultivo}: SHAP de {len(DEV[cultivo])} filas (condado × año), base {BASE[cultivo]:+.1f} bu/ac. Suma verificada.")
    # Ejemplo: Story, Iowa, 2012 (sequía)
    d, s = DEV["maiz"], SHAP["maiz"]
    fila = d.index[(d["fips"] == "19169") & (d["anio"] == 2012)]
    if len(fila):
        print("\nMaíz, Story IA 2012: variables que más pesaron (bu/ac)")
        display(s.loc[fila[0]].sort_values(key=abs, ascending=False).head(6).round(1).to_frame("shap_bu_ac"))
else:
    print("En espera de los datos.")

maiz: SHAP de 11768 filas (condado × año), base -0.2 bu/ac. Suma verificada.


soja: SHAP de 11278 filas (condado × año), base -0.0 bu/ac. Suma verificada.

Maíz, Story IA 2012: variables que más pesaron (bu/ac)


,shap_bu_ac
dias_tmax_gt30_jul,-18.0
tmax_jul_c,-11.0
dias_tmax_gt35_07,-4.7
gwettop_04,4.5
prec_temporada_mm,-4.2
prec_jul_mm,-2.9


## 5. SHAP por grupo de variables

Se suma el SHAP de las variables de cada grupo. Así "calor de julio" junta la temperatura máxima media, los días > 30 °C y los días > 35 °C en un solo número. El resultado se guarda en `backend/Datos/explicaciones_shap_grupos.csv` (una fila por condado, año y cultivo) para el punto 4 y para que el front pueda marcar años en la línea de tiempo.

In [6]:
def columnas_grupo(grupo, cultivo, disponibles):
    cols = [c.replace("<c>", cultivo) for c in GRUPOS[grupo]]
    return [c for c in cols if c in disponibles]


def shap_por_grupo(shap_df, dev, cultivo):
    g = pd.DataFrame(index=shap_df.index)
    for grupo in GRUPOS:
        cols = columnas_grupo(grupo, cultivo, shap_df.columns)
        g[grupo] = shap_df[cols].sum(axis=1) if cols else np.nan
    g.insert(0, "anio", dev["anio"].values)
    g.insert(0, "cultivo", cultivo)
    g.insert(0, "fips", dev["fips"].values)
    return g


GRUPO_SHAP = {}
if HAY_DATOS:
    for cultivo in CULTIVOS:
        GRUPO_SHAP[cultivo] = shap_por_grupo(SHAP[cultivo], DEV[cultivo], cultivo)
    todo_grupos = pd.concat(GRUPO_SHAP.values(), ignore_index=True)
    SALIDA.mkdir(parents=True, exist_ok=True)
    todo_grupos.round(2).to_csv(SALIDA / "explicaciones_shap_grupos.csv", index=False)
    print("Guardado backend/Datos/explicaciones_shap_grupos.csv:", todo_grupos.shape)
    print("\nMaíz, Story IA: SHAP por grupo y año (bu/ac)")
    display(GRUPO_SHAP["maiz"].query("fips == '19169'").set_index("anio").drop(columns=["fips", "cultivo"]).round(1))
else:
    print("En espera de los datos.")

Guardado backend/Datos/explicaciones_shap_grupos.csv: (23046, 13)

Maíz, Story IA: SHAP por grupo y año (bu/ac)


,calor_julio,agua_julio,calor_agosto,agua_agosto,temporada,primavera_humeda,suelo_productivo,agua_del_suelo,terreno,tipo_de_condado
anio,,,,,,,,,,
2009,6.2,2.2,NaN,NaN,2.4,6.2,-1.3,-0.1,-0.1,0.1
2010,4.5,-0.7,NaN,NaN,-12.5,3.2,-1.0,0.0,-0.4,0.4
2011,2.8,1.3,NaN,NaN,2.1,-0.8,-1.1,0.1,-0.1,0.7
2012,-33.7,-4.0,NaN,NaN,-3.7,9.3,-1.6,-0.1,-0.2,1.1
2013,3.4,-5.0,NaN,NaN,2.1,-2.2,-1.8,0.0,-0.1,-0.8
2014,4.5,-3.2,NaN,NaN,1.1,3.8,-1.4,0.0,-0.3,0.6
2015,3.9,1.6,NaN,NaN,0.6,-1.4,-1.2,0.1,-0.2,0.6
2016,4.0,1.9,NaN,NaN,3.4,3.6,-0.9,0.1,-0.1,0.5
2017,1.6,-4.4,NaN,NaN,-1.7,2.9,-1.3,-0.0,-0.1,-0.9


## 6. Resumen por condado: qué pesa en los años malos y qué es fijo

Para cada condado y cultivo:

- **Suelo y terreno** (grupos fijos): el SHAP casi no cambia entre años, así que se toma el promedio. Dice si el suelo del condado empuja el rinde para arriba o para abajo comparado con el condado promedio de la región.
- **Clima** (grupos que cambian cada año): para cada factor se miran sus **años malos**, el 30 % de años en que ese factor más restó. El promedio del SHAP en esos años dice cuánto suele costar ese factor cuando se pone en contra. Se guardan los años para que el front pueda llevar a la línea de tiempo.
- Si el año malo fue por **seco o húmedo, caluroso o fresco**: se compara la variable representativa del grupo en esos años con la media del condado.
- **Qué mueve más el rango:** el grupo con mayor variación entre años, y entre qué valores se movió.

Nada de esto es una causa: son asociaciones en estos datos.

In [7]:
def calificativo(grupo, dev_c, anios_malos):
    """'hot'/'cool' o 'dry'/'wet' según cómo estuvo la variable representativa en los años malos."""
    var, tipo = REPRESENTANTE[grupo]
    if var not in dev_c.columns or dev_c[var].isna().all():
        return None
    media = dev_c[var].mean()
    en_malos = dev_c.loc[dev_c["anio"].isin(anios_malos), var].mean()
    if np.isnan(en_malos):
        return None
    if tipo == "calor":
        return "hot" if en_malos > media else "cool"
    # agua: deficit = lluvia − evapotranspiración (menor = más seco); prec (menor = más seco)
    return "dry" if en_malos < media else "wet"


def resumir_condado(fips, cultivo):
    g = GRUPO_SHAP[cultivo]
    dev_c = DEV[cultivo][DEV[cultivo]["fips"] == fips]
    gc = g[g["fips"] == fips].set_index("anio")
    n = len(gc)
    res = {"fips": fips, "cultivo": cultivo, "n_anios": n, "suelo": {}, "clima": {}, "rango": None}
    if n == 0:
        return res
    for grupo in GRUPOS_SUELO:
        if grupo in gc and gc[grupo].notna().any():
            res["suelo"][grupo] = float(gc[grupo].mean())
    k = max(MIN_ANIOS_MALOS, int(round(n * FRACCION_ANIOS_MALOS)))
    for grupo in GRUPOS_CLIMA:
        if grupo not in gc or gc[grupo].isna().all():
            continue
        peores = gc[grupo].nsmallest(k)
        peores = peores[peores < 0]
        if len(peores) < MIN_ANIOS_MALOS:
            continue
        res["clima"][grupo] = {"valor": float(peores.mean()), "anios": [int(a) for a in peores.index],
                               "calificativo": calificativo(grupo, dev_c, list(peores.index))}
    clima_cols = [c for c in GRUPOS_CLIMA if c in gc and gc[c].notna().any()]
    if clima_cols and n >= 2:
        sd = gc[clima_cols].std().dropna()
    else:
        sd = pd.Series(dtype=float)
    if not sd.empty:
        top = sd.idxmax()
        res["rango"] = {"grupo": top, "min": float(gc[top].min()), "max": float(gc[top].max()),
                        "anio_min": int(gc[top].idxmin()), "anio_max": int(gc[top].idxmax())}
    return res


RESUMEN = {}
if HAY_DATOS:
    for cultivo in CULTIVOS:
        for fips in DEV[cultivo]["fips"].unique():
            RESUMEN[(fips, cultivo)] = resumir_condado(fips, cultivo)
    print(f"Resúmenes: {len(RESUMEN)} (condado × cultivo)")
    r = RESUMEN.get(("19169", "maiz"))
    if r:
        print("\nMaíz, Story IA:")
        print("  suelo:", {k: round(v, 1) for k, v in r["suelo"].items()})
        for k, v in r["clima"].items():
            print(f"  {k}: {v['valor']:+.1f} bu/ac en años {v['anios']} ({v['calificativo']})")
        print("  rango:", r["rango"])
else:
    print("En espera de los datos.")

Resúmenes: 1853 (condado × cultivo)

Maíz, Story IA:
  suelo: {'suelo_productivo': -1.2, 'agua_del_suelo': 0.0, 'terreno': -0.2}
  agua_julio: -4.6 bu/ac en años [2020, 2013, 2017, 2012] (dry)
  temporada: -5.4 bu/ac en años [2010, 2012, 2023, 2021] (dry)
  primavera_humeda: -2.1 bu/ac en años [2019, 2013, 2015, 2011] (wet)
  rango: {'grupo': 'calor_julio', 'min': -33.74511814622144, 'max': 6.17900430224528, 'anio_min': 2012, 'anio_max': 2009}


## 7. Plantillas: de números a frases

Una plantilla fija por grupo y por signo. Reglas (`INFO.md` sección 6 y `PUNTO2.md`):

- **Inglés**, palabras simples ("water the soil can hold", no "AWS").
- **Asociación, no causa:** "tends to", "is linked to"; nunca "because".
- **No recomendar:** nada de "recommended", "best option", "you should". La etiqueta "Suggested for your priorities" la pone el front con sus reglas; acá solo va la evidencia.
- **Precisión honesta:** bu/ac y % sin decimales.
- Cada frase lleva su **fuente** y la marca `basis: "model"` ("From NASA data + model"). Las reglas agronómicas (nitrógeno, carbono, erosión) no salen de acá.

La frase de **rotación** sale de `efecto_rotacion.csv` (el % general) y de `escenarios_rotacion_resumen.csv` (cuántos bu/ac cambia en **este** condado). La de `actual` describe cómo rota el condado hoy.

In [8]:
def bu(v):
    """Entero con signo: '+12' o '-25'."""
    return f"{int(round(v)):+d}"


ETIQUETA_GRUPO = {
    "calor_julio": "July heat", "agua_julio": "July rain and water balance", "calor_agosto": "August heat",
    "agua_agosto": "August rain and water balance", "temporada": "season rain and water balance (Apr–Sep)",
    "primavera_humeda": "spring wetness", "suelo_productivo": "how productive the soil is",
    "agua_del_suelo": "water the soil can hold and how it drains", "terreno": "slope and where water collects",
}


def frase_suelo(grupo, v, cultivo):
    c = CULTIVO_EN[cultivo]
    mas = v > 0
    textos = {
        "suelo_productivo": (f"This county's soil is more productive than most in the region: about {bu(v)} bu/ac for {c} in these data.",
                             f"This county's soil is less productive than most in the region: about {bu(v)} bu/ac for {c} in these data."),
        "agua_del_suelo": (f"How much water the soil can hold and how it drains helps here: about {bu(v)} bu/ac for {c}.",
                           f"How much water the soil can hold and how it drains tends to cost about {bu(v)} bu/ac for {c} here."),
        "terreno": (f"Slope and where water collects help here: about {bu(v)} bu/ac for {c} in these data.",
                    f"Slope and where water collects tend to cost about {bu(v)} bu/ac for {c} here."),
    }
    return textos[grupo][0 if mas else 1]


def frase_clima(grupo, v, calif, cultivo):
    c = CULTIVO_EN[cultivo]
    cuando = {
        "calor_julio": {"hot": "a hot July (corn flowering)", "cool": "an unusually cool July", None: "an unfavorable July temperature"},
        "agua_julio": {"dry": "a dry July", "wet": "a very wet July", None: "an unfavorable July water balance"},
        "calor_agosto": {"hot": "a hot August (soybean pod filling)", "cool": "an unusually cool August", None: "an unfavorable August temperature"},
        "agua_agosto": {"dry": "a dry August", "wet": "a very wet August", None: "an unfavorable August water balance"},
        "temporada": {"dry": "little rain from April to September", "wet": "excess rain from April to September", None: "an unfavorable season water balance"},
        "primavera_humeda": {"wet": "a wet spring (planting delays)", "dry": "a dry spring", None: "unusual spring moisture"},
    }[grupo][calif]
    return f"In years with {cuando}, {c} yields here tend to be about {int(round(abs(v)))} bu/ac lower."


def frase_rango(rango, cultivo):
    c = CULTIVO_EN[cultivo]
    def extremo(v, anio):
        # Menos de 0,5 bu/ac se redondea a "+0": se dice "almost no change" en vez de un cero con signo
        return f"almost no change in {anio}" if int(round(v)) == 0 else f"about {bu(v)} bu/ac in {anio}"
    return (f"What moves this county's {c} range the most is {ETIQUETA_GRUPO[rango['grupo']]}: "
            f"from {extremo(rango['min'], rango['anio_min'])} to {extremo(rango['max'], rango['anio_max'])}.")


def frase_rotacion(rotacion, cultivo, pct_normal, delta_bu, pct_general, hay_otras=True):
    c, o = CULTIVO_EN[cultivo], CULTIVO_EN[OTRO_CULTIVO[cultivo]]
    general = f"{c.capitalize()} after {c} tends to yield about {int(round(abs(pct_general)))}% less than {c} after {o} (2009–2023)."
    if rotacion == "actual":
        if not hay_otras:
            # Fuera de la zona maíz-soja el punto 2 no compara rotaciones: no hay "otras dos opciones"
            return (f"About {int(round(pct_normal))}% of this county's {c} follows {c}. "
                    f"Here {c} and soybean are a small share of the cropland, so rotations aren't compared.")
        return (f"About {int(round(pct_normal))}% of this county's {c} follows {c}. "
                f"Keeping that mix is the reference the other two options are compared with.")
    if abs(delta_bu) < MIN_CAMBIO_BU_AC or int(round(delta_bu)) == 0:
        # El condado ya está casi en esa opción: un "+0 bu/ac" confunde más de lo que explica
        return f"{general} This county already grows close to that mix, so the change here is small."
    if rotacion == "rotado":
        return f"{general} Rotating fully is linked to about {bu(delta_bu)} bu/ac here vs. the county's usual mix."
    if rotacion == "continuo":
        return f"{general} Going all {c}-after-{c} is linked to about {bu(delta_bu)} bu/ac here vs. the usual mix."
    raise ValueError(rotacion)


def razon(tipo, grupo, texto, valor, fuente, anios=None):
    return {"type": tipo, "group": grupo, "text": texto, "value_bu_ac": None if valor is None else round(float(valor), 1),
            "years": anios, "basis": "model", "source": fuente, "source_label": FUENTES[fuente]}

## 8. Armar las razones de cada opción y guardar

Para cada condado, cultivo y opción (`actual`, `rotado`, `continuo`):

1. **Primera razón: la rotación**, siempre, porque es lo que distingue a la opción.
2. Después, las razones de **suelo** y de **clima en años malos** ordenadas por tamaño (bu/ac), solo las que superan el umbral. Se cortan en `N_RAZONES`.
3. **`benefits`** son las que suman y **`risks`** las que restan, para las dos listas de la tarjeta del front.
4. **Confianza** (provisoria, hasta acordarla con el equipo): alta si el condado tiene al menos 12 años con rinde y el efecto de la rotación es claro; media para `rotado` y `continuo`, porque el efecto se midió con cambios chicos y se extiende a un cambio total; baja con pocos años.

Salidas:

- `backend/Datos/explicaciones.csv`: una fila por razón (`fips, cultivo, rotacion, orden, tipo, grupo, valor_bu_ac, anios, texto_en, fuente, basis`).
- `backend/Datos/front/explicaciones.json`: lo mismo anidado por condado → cultivo → rotación, con `confidence`, para que el script del front lo pegue en `next/<fips>.json`.

In [9]:
def confianza(n_anios, rotacion, cultivo, fips):
    ic_claro = efecto.loc[cultivo, "ic95_alto"] < 0
    if n_anios < 6:
        return {"level": "low", "reason": f"Only {n_anios} years with yield data for this county, so the range leans on its state's trend."}
    if fips[:2] == "29" and rotacion in ("rotado", "continuo"):
        # Missouri da el efecto al revés en todas las pruebas del punto 2 (CAMBIOS_9_ESTADOS.md, 2.e)
        return {"level": "low", "reason": "In this state the county data don't show the usual rotation effect."}
    if not ic_claro:
        return {"level": "low", "reason": "The rotation effect is not clearly different from zero in these data."}
    if rotacion in ("rotado", "continuo"):
        return {"level": "medium" if n_anios >= 12 else "low",
                "reason": "The rotation effect was measured on small county-level changes and extended to a full switch."}
    return {"level": "high" if n_anios >= 12 else "medium",
            "reason": f"{n_anios} years of county yield data (2009–2023), model beats the trend baseline."}


# El suelo casi no pesa en el modelo desde que la tendencia tiene un nivel propio por condado (punto 2): ese nivel
# absorbe el suelo y el manejo. Se muestra de dos formas (CAMBIOS_9_ESTADOS.md, 2.f):
#   - "county_level" (modelo): cuánto rinde el condado por encima o por debajo de la recta de su estado;
#   - "soil_rule" (regla): si el suelo retiene mucha o poca agua comparado con la región (cuartil de SSURGO).
AGUA_SUELO = tabla_todo.groupby("fips")["aws0_100_mm"].first() if HAY_DATOS else pd.Series(dtype=float)
CORTES_AGUA = AGUA_SUELO.quantile([0.25, 0.75]).to_list() if len(AGUA_SUELO) else [np.nan, np.nan]


def razon_nivel(fips, cultivo, umbral):
    nivel = MODELOS[cultivo]["tendencia"].get("nivel_condado", {}).get(fips)
    if nivel is None or abs(nivel) < umbral:
        return None
    c = CULTIVO_EN[cultivo]
    lado = "above" if nivel > 0 else "below"
    texto = (f"This county's {c} tends to yield about {abs(int(round(nivel)))} bu/ac {lado} its state's trend "
             f"(its soil and how it is farmed, 2009–2023).")
    return razon("county_level", "nivel_condado", texto, nivel, "nass")


def razon_agua_suelo(fips):
    agua = AGUA_SUELO.get(fips)
    if agua is None or np.isnan(agua):
        return None
    if agua >= CORTES_AGUA[1]:
        texto, signo = (f"This soil holds more water than most in the region (about {int(round(agua))} mm in the top meter), "
                        f"which tends to help in dry summers."), 1
    elif agua <= CORTES_AGUA[0]:
        texto, signo = (f"This soil holds less water than most in the region (about {int(round(agua))} mm in the top meter), "
                        f"so dry spells tend to hurt more."), -1
    else:
        return None
    r = razon("soil_rule", "agua_del_suelo", texto, None, "ssurgo")
    r["basis"], r["sign"] = "rule", signo
    return r


def razones_opcion(fips, cultivo, rotacion, res, mediana):
    umbral = UMBRAL_BU_AC[cultivo]
    pct_normal = float(MODELOS[cultivo]["medias"][MODELOS[cultivo]["rotacion"][0]].get(fips, np.nan))
    delta = {r: mediana.get(r, np.nan) - mediana.get("actual", np.nan) for r in ("rotado", "continuo")}
    pct_general = efecto.loc[cultivo, "continuo_vs_rotado_pct"]
    lista = [razon("rotation", "rotacion", frase_rotacion(rotacion, cultivo, pct_normal, delta.get(rotacion, 0.0), pct_general, "rotado" in mediana.index),
                   None if rotacion == "actual" else delta[rotacion], "cdl")]
    candidatas = []
    for grupo, v in res["suelo"].items():
        if abs(v) >= umbral:
            candidatas.append((abs(v), razon("soil", grupo, frase_suelo(grupo, v, cultivo), v, GRUPO_FUENTE[grupo])))
    for grupo, info in res["clima"].items():
        if abs(info["valor"]) >= umbral:
            candidatas.append((abs(info["valor"]), razon("climate_bad_years", grupo,
                               frase_clima(grupo, info["valor"], info["calificativo"], cultivo), info["valor"],
                               GRUPO_FUENTE[grupo], info["anios"])))
    nivel = razon_nivel(fips, cultivo, umbral)
    if nivel:
        candidatas.append((abs(nivel["value_bu_ac"]), nivel))
    candidatas.sort(key=lambda t: -t[0])
    lista += [r for _, r in candidatas[: N_RAZONES - 1]]
    agua = razon_agua_suelo(fips)
    if agua and len(lista) < N_RAZONES:   # la regla entra solo si queda lugar: primero lo que sale del modelo
        lista.append(agua)
    extra = razon("range_driver", res["rango"]["grupo"], frase_rango(res["rango"], cultivo),
                  res["rango"]["max"] - res["rango"]["min"], GRUPO_FUENTE[res["rango"]["grupo"]],
                  [res["rango"]["anio_min"], res["rango"]["anio_max"]]) if res["rango"] else None
    def signo(r):
        if r["type"] == "rotation":
            return {"actual": 0, "rotado": 1, "continuo": -1}[rotacion]
        if r["type"] == "soil_rule":
            return r["sign"]
        return 1 if (r["value_bu_ac"] or 0) > 0 else -1
    return {
        "reasons": lista,
        "benefits": [r for r in lista if signo(r) > 0],
        "risks": [r for r in lista if signo(r) < 0],
        "range_driver": extra,
        "confidence": confianza(res["n_anios"], rotacion, cultivo, fips),
        "n_years": res["n_anios"],
    }


EXPLICACIONES, FILAS = {}, []
if HAY_DATOS:
    for (fips, cultivo), res in RESUMEN.items():
        med = resumen_esc[(resumen_esc["fips"] == fips) & (resumen_esc["cultivo"] == cultivo)].set_index("rotacion")["mediana_clima"]
        if med.empty or res["n_anios"] == 0:
            continue
        for rotacion in [r for r in ("actual", "rotado", "continuo") if r in med.index]:   # fuera de la zona maíz-soja solo hay "actual"
            doc = razones_opcion(fips, cultivo, rotacion, res, med)
            EXPLICACIONES.setdefault(fips, {}).setdefault(cultivo, {})[rotacion] = doc
            for i, r in enumerate(doc["reasons"] + ([doc["range_driver"]] if doc["range_driver"] else []), start=1):
                FILAS.append({"fips": fips, "cultivo": cultivo, "rotacion": rotacion, "orden": i, "tipo": r["type"],
                              "grupo": r["group"], "valor_bu_ac": r["value_bu_ac"],
                              "anios": ";".join(map(str, r["years"])) if r["years"] else "",
                              "texto_en": r["text"], "fuente": r["source"], "fuente_label": r["source_label"],
                              "basis": r["basis"], "confianza": doc["confidence"]["level"]})
    explicaciones_csv = pd.DataFrame(FILAS)
    explicaciones_csv.to_csv(SALIDA / "explicaciones.csv", index=False)
    FRONT.mkdir(parents=True, exist_ok=True)
    doc_front = {
        "version_contrato": "0.2",
        "generado": datetime.datetime.now().isoformat(timespec="seconds"),
        "es_prueba": False,
        "prototype": False,
        "metric": "yield_bu_ac",
        "note": "Associations in county data (2009–2023), not causes. Yield is for the whole county, not a single field.",
        "modelo_origen": {c: MODELOS[c]["origen"] for c in CULTIVOS},
        "counties": EXPLICACIONES,
    }
    with open(FRONT / "explicaciones.json", "w", encoding="utf-8") as f:
        json.dump(doc_front, f, ensure_ascii=False, indent=1)
    print(f"Guardado backend/Datos/explicaciones.csv ({len(explicaciones_csv)} razones) y backend/Datos/front/explicaciones.json "
          f"({len(EXPLICACIONES)} condados)")
    display(explicaciones_csv.groupby(["cultivo", "rotacion"]).size().unstack())
else:
    print("En espera de los datos.")

Guardado backend/Datos/explicaciones.csv (17216 razones) y backend/Datos/front/explicaciones.json (948 condados)


rotacion,actual,continuo,rotado
cultivo,,,
maiz,3759,2486,2486
soja,3579,2453,2453


## 9. Chequeos

Antes de mostrar nada:

1. **SHAP suma a la predicción** (ya se verificó en el paso 4; acá se vuelve a mirar en una muestra).
2. **Palabras prohibidas:** ninguna frase dice "because", "recommend", "best option" ni "you should".
3. **Largo:** ninguna frase pasa de 40 palabras.
4. **Sentido agronómico:** en Story IA 2012 (sequía) el grupo que más resta en maíz es el calor de julio; y la frase de `rotado` suma mientras la de `continuo` resta, en los dos cultivos.
5. **Cobertura:** todos los condados de `escenarios_rotacion_resumen.csv` tienen explicación.

In [10]:
if HAY_DATOS:
    resultados = {}
    textos = explicaciones_csv["texto_en"]
    resultados["sin_palabras_prohibidas"] = not textos.str.lower().str.contains("|".join(map(re.escape, PROHIBIDAS))).any()
    resultados["frases_cortas_(<=40_palabras)"] = int(textos.str.split().str.len().max()) <= 40
    g12 = GRUPO_SHAP["maiz"].query("fips == '19169' and anio == 2012")
    resultados["story_2012_manda_calor_julio"] = (not g12.empty) and g12.iloc[0][GRUPOS_CLIMA].idxmin() == "calor_julio"
    rot = explicaciones_csv[explicaciones_csv["tipo"] == "rotation"]
    resultados["rotado_suma_y_continuo_resta"] = bool(
        (rot[rot["rotacion"] == "rotado"]["valor_bu_ac"] > 0).all() and (rot[rot["rotacion"] == "continuo"]["valor_bu_ac"] < 0).all())
    faltan = set(map(tuple, resumen_esc[["fips", "cultivo"]].drop_duplicates().values)) - set(RESUMEN)
    resultados["todos_los_condados_con_escenario_tienen_explicacion"] = len(faltan) == 0
    resultados["cada_opcion_tiene_de_1_a_N_razones"] = explicaciones_csv[explicaciones_csv["tipo"] != "range_driver"] \
        .groupby(["fips", "cultivo", "rotacion"]).size().between(1, N_RAZONES).all()
    for k, v in resultados.items():
        print(f"{'OK   ' if v else 'FALLA'} {k}")
    if not all(resultados.values()):
        print("ATENCIÓN: algún chequeo falló. Revisar antes de pasar al front.")
    palabras = explicaciones_csv["texto_en"].str.split().str.len()
    print(f"\nLargo de las frases: mediana {int(palabras.median())} palabras, máximo {int(palabras.max())}.")
else:
    print("En espera de los datos.")

OK    sin_palabras_prohibidas
OK    frases_cortas_(<=40_palabras)
OK    story_2012_manda_calor_julio
OK    rotado_suma_y_continuo_resta
OK    todos_los_condados_con_escenario_tienen_explicacion
OK    cada_opcion_tiene_de_1_a_N_razones

Largo de las frases: mediana 21 palabras, máximo 28.


## 10. Ejemplos: lo que vería el productor

Story (Iowa) y McLean (Illinois), los condados de demo con historia completa. Sumner (Kansas) no está en la entrega v1.2 (que cubre Iowa, Illinois e Indiana): la app tiene que mostrar "Missing data" ahí hasta que el pipeline sume Kansas.

In [11]:
def mostrar(fips, cultivo):
    nombre = {"19169": "Story, IA", "17113": "McLean, IL", "20191": "Sumner, KS"}.get(fips, fips)
    print(f"\n==== {nombre} · {CULTIVO_EN[cultivo]}")
    doc = EXPLICACIONES.get(fips, {}).get(cultivo)
    if not doc:
        print("   Missing data: this county is not in the data delivery.")
        return
    for rotacion, d in doc.items():
        print(f"  [{rotacion}]  confidence: {d['confidence']['level']} — {d['confidence']['reason']}")
        for r in d["reasons"]:
            print(f"     • {r['text']}  ({r['source_label']})")
        if d["range_driver"]:
            print(f"     ~ {d['range_driver']['text']}  ({d['range_driver']['source_label']})")


if HAY_DATOS:
    for fips in ["19169", "17113", "20191"]:
        for cultivo in CULTIVOS:
            mostrar(fips, cultivo)
else:
    print("En espera de los datos.")


==== Story, IA · corn
  [actual]  confidence: high — 15 years of county yield data (2009–2023), model beats the trend baseline.
     • About 35% of this county's corn follows corn. Keeping that mix is the reference the other two options are compared with.  (USDA CDL + NASS)
     • In years with little rain from April to September, corn yields here tend to be about 5 bu/ac lower.  (NASA POWER)
     • In years with a dry July, corn yields here tend to be about 5 bu/ac lower.  (NASA POWER)
     ~ What moves this county's corn range the most is July heat: from about -34 bu/ac in 2012 to about +6 bu/ac in 2009.  (NASA POWER)
  [rotado]  confidence: medium — The rotation effect was measured on small county-level changes and extended to a full switch.
     • Corn after corn tends to yield about 19% less than corn after soybean (2009–2023). Rotating fully is linked to about +12 bu/ac here vs. the county's usual mix.  (USDA CDL + NASS)
     • In years with little rain from April to September, 

## Próximos pasos

- [ ] Acordar con el front dónde pegar `backend/Datos/front/explicaciones.json` en `next/<fips>.json` (`benefits`, `risks`, `confidence`, `years[]` para F23) y confirmar que los ids `actual`/`rotado`/`continuo` coinciden con los de las opciones.
- [ ] Acordar con el equipo la regla de confianza (hoy: años con dato + claridad del efecto de rotación + si la opción extrapola).
- [ ] Cuando el pipeline sume Kansas (Sumner `20191`), volver a correr el punto 2 y esta notebook.
- [ ] Revisar con alguien de agronomía las frases de `temporada` y `primavera_humeda`: el calificativo seco/húmedo sale de una sola variable representativa.
- [ ] Si el modelo elegido del punto 2 pasa a ser Ridge, cambiar `shap_del_modelo` a coeficiente × valor escalado.
- [ ] Sumar las frases de `efecto_rotacion.csv` a `textos.json` cuando el front lo tenga.
- [ ] `docs/ml/model_card.md`: sección de explicabilidad.